# **Import the neccessary libray**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# **load the data**

In [ ]:
file_path = input("enter the location of the file ")
df_lap = pd.read_csv(f"{file_path}\\f1_lap_data.csv")
df_race = pd.read_csv(f"{file_path}\\f1_race_data.csv")

# **EDA**

## EDA for lap data

- Lap data information

In [ ]:
df_lap.info()

- First 5 row of the data

In [ ]:
df_lap.head()

- lap data shape

In [ ]:
df_lap.shape

- Duplicates in the data

In [ ]:
df_lap.duplicated().sum()

- null values

In [ ]:
print(df_lap.isna().sum())
print(df_lap.isnull().mean().mul(100).round(2))

## EDA for race data

- Information on race data

In [ ]:
df_race.info()

- Race data shape

In [ ]:
df_race.shape

- First 5 row of the data

In [ ]:
df_race.head()

- Duplicates in the data

In [ ]:
df_race.duplicated().sum()

- Null/NaN values

In [ ]:
print(df_race.isna().sum())
print(df_race.isnull().mean().mul(100).round(2))

# **Data Cleaning**

- Datetime correction

In [ ]:
df_race['race_date'] = pd.to_datetime(df_race['race_date'])

- Handle NaN value

In [ ]:
df_lap['lap_time_seconds'] = df_lap['lap_time_seconds'].fillna(df_lap['lap_time_seconds'].median())
df_race['race_time_seconds'] = df_race['race_time_seconds'].fillna(df_race['race_time_seconds'].median())
df_race['avg_lap_time_seconds'] = df_race['avg_lap_time_seconds'].fillna(df_race['avg_lap_time_seconds'].median())

- Handle duplicates

In [ ]:
df_lap.drop_duplicates(inplace=True)
df_race.drop_duplicates(inplace=True)

## *Add New Columns*

In [ ]:
df_race["positions_gained"]   = df_race["grid_position"] - df_race["finish_position"]
df_race["race_time_minutes"]  = (df_race["race_time_seconds"] / 60).round(2)
df_race["is_points_finish"]   = df_race["finish_position"] <= 10

df_race.head()


## *Export the clean data*

In [ ]:
file_path = input("enter the location you want the file to store")
df_lap.to_csv(f"{file_path}\\f1_lap__cleaned.csv",index=False)
df_race.to_csv(f"{file_path}\\f1_race__cleaned.csv",index=False)

# **Data Analysis**

## *Driver summery*

In [ ]:
df_driver_summary = df_race.groupby("driver_name").agg(
    total_points       = ("points", "sum"),
    total_races        = ("race_id", "count"),
    total_wins         = ("finish_position", lambda x: (x == 1).sum()),
    avg_finish_position= ("finish_position", "mean")
).round(2).sort_values("total_points", ascending=False)

df_driver_summary

## *Team summery*

In [ ]:
df_team_summary = df_race.groupby(["team", "season"]).agg(
    total_points        = ("points", "sum"),
    total_wins          = ("finish_position", lambda x: (x == 1).sum()),
    avg_finish_position = ("finish_position", "mean")
).round(2).sort_values(["season", "total_points"], ascending=[True, False])

df_team_summary


## *Pit Stop Analysis*

In [ ]:
# Avg pit stops per team
df_race.groupby("team")["pit_stops"].mean().round(2).sort_values(ascending=False)

In [ ]:
# Driver with most total pit stops
df_race.groupby("driver_name")["pit_stops"].sum().sort_values(ascending=False).head(1)

In [ ]:
# Avg pit stops by weather
df_race.groupby("weather")["pit_stops"].mean().round(2)

## *Fastest Lap Analysis*

In [ ]:
# How many fastest laps each driver has earned in total
df_race.groupby('driver_name')['fastest_lap'].sum().sort_values(ascending=False)

In [ ]:
# Which team has earned the most fastest laps
df_race.groupby('team')['fastest_lap'].sum()

In [ ]:
# Percentage of races with a fastest lap awarded
pct = df_race["fastest_lap"].mean() * 100
print(f"Races with fastest lap awarded: {pct:.2f}%")

## *DNF (Did Not Finish) Analysis*

In [ ]:
# DNF count per driver
df_race.groupby("driver_name")["dnf"].sum().sort_values(ascending=False)

In [ ]:
# DNF rate per team
df_race.groupby("team").agg(
    dnf_rate=("dnf", lambda x: round(x.mean() * 100, 2))
).sort_values("dnf_rate", ascending=False)

In [ ]:
# Circuit with highest DNF rate
df_race.groupby("circuit_name").agg(
    dnf_rate=("dnf", lambda x: round(x.mean() * 100, 2))
).sort_values("dnf_rate", ascending=False)

## *Season Championship Standings*

In [ ]:
df_standings = df_race.groupby(["season", "driver_name"]).agg(
    total_points = ("points", "sum"),
    total_wins   = ("finish_position", lambda x: (x == 1).sum())
).reset_index()

df_standings["rank"] = df_standings.groupby("season")["total_points"]\
                                   .rank(ascending=False, method="min").astype(int)

df_standings.sort_values(["season", "rank"])